In [20]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

In [21]:
df=pd.read_csv('BTC-USD1d.csv').dropna()
df.drop(index=0, inplace=True)
df['Date'] = pd.to_datetime(df['Date'])
df

,Date,Close,High,Low,Open,Volume
1,2024-01-02,44957.96875,45899.70703,44176.94922,44187.14063,3.933527e+10
2,2024-01-03,42848.17578,45503.24219,40813.53516,44961.60156,4.634232e+10
3,2024-01-04,44179.92188,44770.02344,42675.17578,42855.81641,3.044809e+10
4,2024-01-05,44162.69141,44353.28516,42784.71875,44192.98047,3.233603e+10
5,2024-01-06,43989.19531,44227.63281,43475.15625,44178.95313,1.609250e+10
...,...,...,...,...,...,...
361,2024-12-27,94164.85938,97294.84375,93310.74219,95704.97656,5.241993e+10
362,2024-12-28,95163.92969,95525.89844,94014.28906,94160.18750,2.410744e+10
363,2024-12-29,93530.22656,95174.87500,92881.78906,95174.05469,2.963589e+10
364,2024-12-30,92643.21094,94903.32031,91317.13281,93527.19531,5.618800e+10


Purpose of Multiplying by 0.05:
1.Scaling Volatility Impact: Volatility, represented by the standard deviation of price changes, can be quite large in absolute terms, especially in highly volatile markets like cryptocurrencies. Multiplying by 0.05 scales this volatility to a small fraction of the overall price range, ensuring that the threshold adjustment is subtle yet meaningful.
2.Avoiding Over-Sensitivity: Without this scaling, the volatility factor might make the threshold overly sensitive to price movements. For instance, if you directly used the volatility without scaling, even minor fluctuations could significantly alter the threshold, potentially leading to too many detected levels or too few, depending on the market's state.
3.Maintaining a Baseline: The base value in the calculation is 1, which represents no change from the current price. By adding a small fraction of the volatility, we're adjusting this base slightly, ensuring that the threshold still primarily reflects the price level but with a nuanced adjustment for market conditions.

In [22]:

scaler = StandardScaler()
df[['High1', 'Low1']] = scaler.fit_transform(df[['High', 'Low']])

def dynamic_sensitivity(df, lookback=20):
    # Calculate rolling standard deviation as a proxy for volatility
    volatility = df['High1'].rolling(window=lookback).std()
    # Dynamic DS based on current volatility
    return 1 + (volatility / df['High1'].max()) * 0.05  # Adjust factor as needed

DS=dynamic_sensitivity(df)
# DS (Dynamic Sensitivity) is a measure that adjusts based on the volatility of the 'High' prices.
# It is calculated as 1 plus a fraction of the rolling standard deviation of 'High' prices over a specified lookback period.
# This fraction is determined by dividing the rolling standard deviation by the maximum 'High' price and multiplying by a factor (0.05 in this case).
# The purpose of DS is to dynamically adjust the sensitivity of support and resistance levels based on market volatility.


def is_support(df, i):
    support = (df['Low'].iloc[i] * DS.iloc[i] < df['Low'].iloc[i-1] * DS.iloc[i-1]) and \
              (df['Low'].iloc[i] * DS.iloc[i] < df['Low'].iloc[i+1] * DS.iloc[i+1]) and \
              (df['Low'].iloc[i+1] * DS.iloc[i+1] < df['Low'].iloc[i+2] * DS.iloc[i+2]) and \
              (df['Low'].iloc[i-1] * DS.iloc[i-1] < df['Low'].iloc[i-2] * DS.iloc[i-2])
    return support

def is_resistance(df, i):
    resistance = (df['High'].iloc[i] > df['High'].iloc[i-1] * DS.iloc[i-1]) and \
                 (df['High'].iloc[i] > df['High'].iloc[i+1] * DS.iloc[i+1]) and \
                 (df['High'].iloc[i+1] > df['High'].iloc[i+2] * DS.iloc[i+2]) and \
                 (df['High'].iloc[i-1] > df['High'].iloc[i-2] * DS.iloc[i-2])
    return resistance



In [23]:
# Reset the index so that the rows are numbered consecutively
df.reset_index(drop=True, inplace=True)

support=[]
resistance=[]

for i in range(2,len(df['Low'])-2):
    if is_support(df,i):
        # to avoid repitition
        if not support or abs(df['Low'][i] - support[-1][1]) > df['Low'][i]*0.02:
            support.append((i,df['Low'][i]))

    elif is_resistance(df,i):
        if not resistance or abs(df['High'][i] - resistance[-1][1]) > df['High'][i]*0.02:
            resistance.append((i,df['High'][i]))

# Bundle close support and resistance levels
def bundle_levels(levels, tolerance):
    bundled_levels = []
    i = 0
    while i < len(levels):
        current_level = levels[i]
        bundle = [current_level]
        j = i + 1
        while j < len(levels) and abs(levels[j][1] - current_level[1]) <= tolerance:
            bundle.append(levels[j])
            j += 1
        
        # Calculate average value for the bundle
        avg_value = sum(level[1] for level in bundle) / len(bundle)
        
        # Use the index of the first level in the bundle
        bundled_levels.append((bundle[0][0], avg_value))
        
        i = j
    return bundled_levels

# Define a tolerance for bundling (e.g., 2% of the price)
price_range = df['High'].max() - df['Low'].min()
tolerance_percentage = 0.05
tolerance = price_range * tolerance_percentage

bundled_support = bundle_levels(support, tolerance)
bundled_resistance = bundle_levels(resistance, tolerance)

len(support), len(bundled_support), len(resistance), len(bundled_resistance)

(20, 15, 11, 10)

In [26]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Convert 'Price', 'Close', 'High', 'Low', 'Open' to numeric, coercing errors to NaN
for col in ['Close', 'High', 'Low', 'Open']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop rows with NaN values introduced by the conversion
df = df.dropna()

# Convert 'Price' column to datetime objects
df['Date'] = pd.to_datetime(df['Date'])

# Create subplots
fig = make_subplots(rows=1, cols=1, shared_xaxes=True, vertical_spacing=0.03,
                    row_width=[1], column_width=[1])

# Add candlestick trace
fig.add_trace(go.Candlestick(x=df['Date'], open=df['Open'], high=df['High'], low=df['Low'], close=df['Close'], name='Candlestick'), row=1, col=1)

# Add support and resistance lines
for res in bundled_resistance:
    fig.add_trace(go.Scatter(x=[df['Date'].iloc[res[0]], df['Date'].iloc[-1]], y=[res[1], res[1]], mode='lines', line=dict(color='red'), name='Resistance'))

for sup in bundled_support:
    fig.add_trace(go.Scatter(x=[df['Date'].iloc[sup[0]], df['Date'].iloc[-1]], y=[sup[1], sup[1]], mode='lines', line=dict(color='green'), name='Support'))

fig.update_layout(
    title='BTC-USD Candlestick Chart with Support and Resistance',
    xaxis_title='Date',
    yaxis_title='Price',
    xaxis_rangeslider_visible=False
)

fig.show()
